In [8]:
import pandas as pd
import holidays

def load_daily_exits(path):
    df = pd.read_csv(
        path,
        names=["date", "hour", "origin", "dest", "trips"],
        usecols=["date", "dest", "trips"],          # skip columns we don't need
        dtype={"dest": "category", "trips": "int32"},
    )
    return df.groupby(["date", "dest"], observed=True)["trips"].sum().reset_index()

daily = pd.concat([
    load_daily_exits("../data/raw/date-hour-soo-dest-2024.csv.gz"),
    load_daily_exits("../data/raw/date-hour-soo-dest-2025.csv.gz"),
])
daily = daily.rename(columns={"dest": "station", "trips": "exits"})
daily["date"] = pd.to_datetime(daily["date"])
daily.head()

,date,station,exits
0,2024-01-01,12TH,1181
1,2024-01-01,16TH,1994
2,2024-01-01,19TH,1291
3,2024-01-01,24TH,1877
4,2024-01-01,ANTC,347


In [2]:
top = daily.groupby("station", observed=True)["exits"].sum().nlargest(8).index
daily = daily[daily["station"].isin(top)].copy()
daily["station"] = daily["station"].astype(str)
print(list(top))

['EMBR', 'MONT', 'POWL', 'CIVC', 'DBRK', '16TH', '24TH', '12TH']


In [3]:
weather = pd.read_csv("../data/raw/weather.csv", parse_dates=["date"])
df = daily.merge(weather, on="date", how="left")
df.isna().sum()   # should be all zeros

date          0
station       0
exits         0
temp_max_f    0
precip_in     0
dtype: int64

In [5]:
us_holidays = holidays.US(years=[2024, 2025])

df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month
df["is_holiday"] = df["date"].dt.date.isin(us_holidays).astype(int)

df.sample(10)

,date,station,exits,temp_max_f,precip_in,day_of_week,month,is_holiday
1049,2024-05-11,16TH,4334,76.8,0.000,Saturday,5,0
5201,2025-10-12,16TH,3158,71.2,0.000,Sunday,10,0
2842,2024-12-21,24TH,3332,59.1,0.201,Saturday,12,0
91,2024-01-12,CIVC,7428,52.4,0.000,Friday,1,0
1316,2024-06-13,DBRK,6720,66.6,0.000,Thursday,6,0
1156,2024-05-24,DBRK,4505,60.4,0.000,Friday,5,0
1320,2024-06-14,12TH,4148,80.7,0.000,Friday,6,0
1339,2024-06-16,CIVC,3555,84.5,0.000,Sunday,6,0
1104,2024-05-18,12TH,3030,65.6,0.000,Saturday,5,0
2663,2024-11-28,POWL,3842,56.8,0.000,Thursday,11,1


In [6]:
df.to_csv("../data/processed/bart_daily.csv", index=False)
print(df.shape)

(5848, 8)


In [15]:
import sys
sys.path.append("../src")

import pandas as pd
from linreg.preprocessing import split_by_date

df = pd.read_csv("../data/processed/bart_daily.csv")
train, test = split_by_date(df)

print(train['date'].max(), test['date'].min())
print(len(train) + len(test) == len(df))


KeyError: 'data'